# Filters, a trainable CNN and mask metrics · Modules 9, 11–12
Python 3 and NumPy only. The code uses valid unflipped cross-correlation and row-batch, channel-last activations. No GPU or framework is required. Derive local filter responses and the network backward equations first.

The synthetic classification and segmentation tasks establish mechanics for their generators. They are not benchmark detection or segmentation systems.


In [1]:
import numpy as np
np.set_printoptions(precision=4,suppress=True)
rng=np.random.default_rng(209)
def correlate(image,kernel):
    kh,kw=kernel.shape;h,w=image.shape
    out=np.zeros((h-kh+1,w-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]): out[i,j]=np.sum(image[i:i+kh,j:j+kw]*kernel)
    return out
image=np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
kernel=np.array([[1.,0.],[0.,-1.]])
assert np.allclose(correlate(image,kernel),-4)
print('valid cross-correlation:\n',correlate(image,kernel))
print('valid flipped convolution:\n',correlate(image,kernel[::-1,::-1]))
row=np.array([1.,-1.,1.,-1.])
spectrum=np.fft.fft(row)
assert np.allclose(spectrum,[0,0,4,0])
assert np.allclose(np.fft.ifft(spectrum).real,row)
print('alternating-row DFT:',spectrum)
print('output length L7,K3,P1,S2:',(7+2-3)//2+1)


valid cross-correlation:
 [[-4. -4.]
 [-4. -4.]]
valid flipped convolution:
 [[4. 4.]
 [4. 4.]]
alternating-row DFT: [0.+0.j 0.+0.j 4.+0.j 0.+0.j]
output length L7,K3,P1,S2: 4


## Independent bars and a fixed protocol
Generate 160 training, 40 validation and 40 test images. The declared CNN has four 3x3 kernels, ReLU and a two-class flattened head. It trains all convolution and head parameters for 300 full-batch steps at learning rate 0.1. The test set is inspected only after this fixed procedure.


In [2]:
def make_bars(n):
    y=rng.integers(0,2,n);images=np.zeros((n,8,8))
    for i,label in enumerate(y):
        position=int(rng.integers(1,6))
        if label==0: images[i,position:position+2,:]=1
        else: images[i,:,position:position+2]=1
    images+=rng.normal(0,.1,images.shape)
    return images,y
X,y=make_bars(240)
Xtr,Xval,Xtest=X[:160],X[160:200],X[200:]
ytr,yval,ytest=y[:160],y[160:200],y[200:]
def patches(X):
    return np.lib.stride_tricks.sliding_window_view(X,(3,3),axis=(1,2)).reshape(len(X),6,6,9)
Ptr,Pval,Ptest=patches(Xtr),patches(Xval),patches(Xtest)
parameters={'K':rng.normal(0,.25,(9,4)),'bc':np.zeros(4),
            'W':rng.normal(0,.05,(144,2)),'b':np.zeros(2)}
def cnn(P,p):
    Z=np.einsum('nijk,kf->nijf',P,p['K'])+p['bc']
    A=np.maximum(0,Z).reshape(len(P),-1)
    logits=A @ p['W']+p['b']
    return Z,A,logits
def loss_grad(P,y,p):
    Z,A,logits=cnn(P,p)
    shifted=logits-logits.max(axis=1,keepdims=True)
    ex=np.exp(shifted);probs=ex/ex.sum(axis=1,keepdims=True)
    loss=np.mean(np.log(ex.sum(axis=1))-shifted[np.arange(len(y)),y])
    G=(probs-np.eye(2)[y])/len(y)
    Gconv=(G @ p['W'].T).reshape(Z.shape)*(Z>0)
    gradients={'W':A.T @ G,'b':G.sum(axis=0),
               'K':np.einsum('nijk,nijf->kf',P,Gconv),
               'bc':Gconv.sum(axis=(0,1,2))}
    return loss,gradients
assert cnn(Ptr[:2],parameters)[0].shape==(2,6,6,4)
print('trainable parameters:',sum(v.size for v in parameters.values()))


trainable parameters: 330


## Check a shared-kernel derivative
The kernel gradient accumulates contributions over the batch and all placements. All derivatives are computed using the original forward parameters.


In [3]:
_,analytic=loss_grad(Ptr[:4],ytr[:4],parameters)
for name,index in [('K',(1,2)),('bc',(1,)),('W',(17,0)),('b',(0,))]:
    value=parameters[name][index];h=1e-5
    parameters[name][index]=value+h;plus=loss_grad(Ptr[:4],ytr[:4],parameters)[0]
    parameters[name][index]=value-h;minus=loss_grad(Ptr[:4],ytr[:4],parameters)[0]
    parameters[name][index]=value
    numeric=(plus-minus)/(2*h);error=abs(numeric-analytic[name][index])
    print('CNN gradient check:',name,index,error)
    assert error<1e-6
initial=loss_grad(Ptr,ytr,parameters)[0]
for step in range(300):
    loss,gradients=loss_grad(Ptr,ytr,parameters)
    for name in parameters: parameters[name]-=.1*gradients[name]
final=loss_grad(Ptr,ytr,parameters)[0]
assert np.isfinite(final) and final<initial
print('CNN initial/final training loss:',initial,final)
print('validation accuracy:',np.mean(cnn(Pval,parameters)[2].argmax(axis=1)==yval))
print('fixed-procedure test accuracy:',np.mean(cnn(Ptest,parameters)[2].argmax(axis=1)==ytest),'N:',len(ytest))


CNN gradient check: K (1, 2) 4.769919268593492e-12
CNN gradient check: bc (1,) 8.274131380048289e-12
CNN gradient check: W (17, 0) 2.433470092100265e-13
CNN gradient check: b (0,) 5.450057072309278e-12
CNN initial/final training loss: 0.723314126303416 0.0029355229308420986
validation accuracy: 1.0
fixed-procedure test accuracy: 1.0 N: 40


## Box and ranking metrics
Boxes use continuous boundary coordinates without +1. This AP example is noninterpolated toy AP at one fixed matching threshold, not COCO AP.


In [4]:
def box_iou(a,b):
    left=np.maximum(a[:2],b[:2]);right=np.minimum(a[2:],b[2:])
    wh=np.maximum(0,right-left);intersection=np.prod(wh)
    union=np.prod(a[2:]-a[:2])+np.prod(b[2:]-b[:2])-intersection
    return intersection/union if union>0 else 0.
value=box_iou(np.array([0.,0.,2.,2.]),np.array([1.,1.,3.,3.]))
assert np.isclose(value,1/7)
print('box IoU:',value)
tp=np.array([1,0,1]);precision=np.cumsum(tp)/np.arange(1,4)
ap=precision[tp==1].sum()/2
assert np.isclose(ap,5/6)
print('ranked precision:',precision,'noninterpolated toy AP:',ap)


box IoU: 0.14285714285714285
ranked precision: [1.     0.5    0.6667] noninterpolated toy AP: 0.8333333333333333


## A trainable pixel classifier for rectangles
One scalar logit per pixel models bright synthetic foreground. This is deliberately simpler than an encoder-decoder segmentation network. Train and select on separate images, then compare foreground IoU with the all-background baseline.


In [5]:
def rectangles(n):
    masks=np.zeros((n,8,8),dtype=bool)
    for i in range(n):
        top,left=rng.integers(0,4,2);height,width=rng.integers(2,5,2)
        masks[i,top:top+height,left:left+width]=True
    observed=.8*masks+rng.normal(0,.12,masks.shape)
    return observed,masks
observed,masks=rectangles(140)
train,validation,test=observed[:100],observed[100:120],observed[120:]
ttr,tval,ttest=masks[:100],masks[100:120],masks[120:]
a=1.;bias=0.
for _ in range(500):
    z=a*train+bias;p=1/(1+np.exp(-z))
    residual=p-ttr
    da=np.mean(residual*train);db=np.mean(residual)
    a-=.8*da;bias-=.8*db
def mask_metrics(pred,true):
    tp=np.sum(pred & true);fp=np.sum(pred & ~true);fn=np.sum(~pred & true)
    union=tp+fp+fn
    iou=tp/union if union else 0. # Explicit empty-union convention.
    dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    return dict(accuracy=float(np.mean(pred==true)),iou=float(iou),dice=float(dice),tp=int(tp),fp=int(fp),fn=int(fn))
print('pixel classifier parameters:',a,bias)
print('validation:',mask_metrics(a*validation+bias>=0,tval))
prediction=a*test+bias>=0
print('test all-background:',mask_metrics(np.zeros_like(ttest),ttest))
print('test learned mask:',mask_metrics(prediction,ttest))
assert prediction.shape==ttest.shape
assert mask_metrics(np.zeros_like(ttest),ttest)['iou']==0


pixel classifier parameters: 8.979170370217588 -4.478488358451967
validation: {'accuracy': 1.0, 'iou': 1.0, 'dice': 1.0, 'tp': 169, 'fp': 0, 'fn': 0}
test all-background: {'accuracy': 0.87109375, 'iou': 0.0, 'dice': 0.0, 'tp': 0, 'fp': 0, 'fn': 165}
test learned mask: {'accuracy': 1.0, 'iou': 1.0, 'dice': 1.0, 'tp': 165, 'fp': 0, 'fn': 0}


## Capstone extension
Declare one new comparison before training: change bar noise, training augmentation, kernel count or rectangle contrast. Keep target transformations aligned with input transformations. Use fresh held-out data for a revised procedure. Save three failures or clearly labeled stressful cases; report coordinates, counts and metric conventions. Complete the capstone template with a baseline, one controlled improvement, reproducible settings and limits of the evidence.
